# Agentic AI Hardening - The Infinite Loop Problem

## Introduction to LangGraph
**LangGraph** is a library built on top of `LangChain` for creating stateful, multi-agent applications. While standard LLM chains are typically directed acyclic graphs (`DAGs`), `LangGraph` is distinct because it allows **cycles**. 

This cyclic capability is essential for building sophisticated AI agents that can reason, act, observe their own output, and loop back to refine their work based on feedback or environmental changes.

### Core Concepts
1. **State**: A persistent object that represents the current memory or status of the workflow. Every node in the graph receives the state, modifies it, and passes it on.
2. **Nodes**: Python functions that encapsulate logic, such as calling an LLM or interacting with a database.
3. **Edges**: Transition logic between nodes. Normal edges provide direct, deterministic transitions from one step to the next. Conditional edges provide logic-based routing that decides which node to visit next based on values within the **State**.

## Lab Objectives
This section explores the problems of cyclic autonomy. The objectives are to:
- Understand the basic mechanics of a `LangGraph` `StateGraph`.
- Witness how autonomous agents can accidentally enter **Infinite Loops** (Perpetual Refinement).
- Implement **Iterative Guardrails** (Maximum Iterations) to ensure system stability, predictability, and cost control.

## Part 1: The Simple Self-Loop

Begin with the most basic cyclic structure possible: a single node that points directly back to itself. This example demonstrates the fundamental syntax of `LangGraph` and highlights the immediate risk: a loop with no exit condition runs until external resources are exhausted or the process is killed.

> **Tip:** This cell loops indefinitely. Stop the kernel manually after observing several iterations.

In [ ]:
from langgraph.graph import StateGraph
import time

# Define a simple state (can be anything, dict is common)
class State(dict):
    pass

# Define a node function
def loop_node(state: State):
    print("Looping...")
    time.sleep(2)
    return state

# Build the graph
builder = StateGraph(State)

builder.add_node("loop", loop_node)

# Add an edge that loops back to itself
builder.add_edge("loop", "loop")

# Set entry point
builder.set_entry_point("loop")

graph = builder.compile()

# Run (⚠️ this will loop forever)
graph.invoke({})

## Part 2: The "Confused Evaluator" Problem

In real-world Agentic AI, infinite loops are rarely as obvious as the example above. They often emerge from the interaction between two specialized nodes:
1. **The Worker Node**: An agent tasked with performing a task or improving a result.
2. **The Judge/Evaluator Node**: An agent tasked with deciding whether the work is complete or good enough.

If the worker's improvements are marginal and the judge's standards are extremely high, or the prompts are ambiguous, the system can enter a perpetual refinement state. The agents appear to make progress, but the system is actually spinning, consuming tokens and compute time without reaching a conclusion.

> **Tip:** This lab does not use the most powerful available model. The judge should not consider the task complete because the task asks for continuous improvement. However, LLM non-determinism can cause the judge to stop the loop. If that occurs, rerun the cell until the loop behavior appears, then stop the kernel after a few iterations.

> **Question:** Which part of this graph makes termination depend on model output rather than deterministic control logic?

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

# -----------------------
# LLM (local server)
# -----------------------
llm = ChatOpenAI(
    model="qwen3.5-2b:Q4_K_M:no-thinking",
    api_key="llama-cpp",
    base_url="http://llama-cpp-qwen3-5-2b.llama-cpp:8080/v1/"
)

# -----------------------
# State
# -----------------------
class State(TypedDict):
    task: str
    result: str
    judgment: str  # YES / NO

# -----------------------
# TASK 1: Agent produces work
# -----------------------
def task_1_agent(state: State):
    prompt = f"""
You are performing a task.

Task: {state['task']}
Current result: {state['result']}

Improve or advance the result.
Return ONLY the updated result.
"""

    print("calling LLM model...")
    response = llm.invoke(prompt)
    print(f"result: {response.content}")
    return {"result": response.content}

# -----------------------
# TASK 2: Agent judges completion
# -----------------------
def task_2_judge_agent(state: State):
    prompt = f"""
You are a strict evaluator.

Task: {state['task']}
Result: {state['result']}

Is the task COMPLETE?

Think carefully, but respond ONLY with YES or NO.
"""

    response = llm.invoke(prompt)
    print(f"judgement: {response.content}")
    return {"judgment": response.content.strip().upper()}


# -----------------------
# ROUTING LOGIC
# -----------------------
def route_after_tool(state: State):
    last_output = state.get("judgment", "")

    if "YES" in last_output:
        return END
    return "task_1"

# -----------------------
# BUILD GRAPH
# -----------------------
builder = StateGraph(State)

builder.add_node("task_1", task_1_agent)
builder.add_node("task_2", task_2_judge_agent)

builder.set_entry_point("task_1")

# Flow:
builder.add_edge("task_1", "task_2")

builder.add_conditional_edges(
    "task_2",
    route_after_tool,
    {
        "task_1": "task_1",
        END: END
    }
)

graph = builder.compile()

# -----------------------
# RUN
# -----------------------
graph.invoke({
    "task": "Keep improving the explanation of 'why LLMs hallucinate' forever. Make it clearer each time, it's never good enough.",
    "result": "LLMs hallucinate becuase of its training",
    "judgment": ""
})

## Part 3: Implementing Secure Guardrails

In any Zero-Trust or production-grade AI system, process termination cannot rely solely on the LLM's judgment. The system must implement **deterministic guardrails**.

The most fundamental safety mechanism is the **Iteration Limit**. By tracking the number of cycles within the **State** and enforcing a hard exit, the system remains stable even if the underlying agents fail to reach agreement.

### Key Changes
1. **State Extension**: Add an `iteration` counter to the global state object.
2.  **Counter Logic**: The worker node now increments the `iteration` count with every pass.
3. **Hard Exit Logic**: The **Conditional Edge** checks whether `iteration >= MaxIter` *before* evaluating the LLM's judgment. If the limit is reached, it forces the graph to `END`.

> **Question:** What operational risks does an iteration limit reduce besides infinite loops?

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

MaxIter = 1

# -----------------------
# LLM (local server)
# -----------------------
llm = ChatOpenAI(
    model="qwen3.5-2b:Q4_K_M:no-thinking",
    api_key="llama-cpp",
    base_url="http://llama-cpp-qwen3-5-2b.llama-cpp:8080/v1/"
)

# -----------------------
# State
# -----------------------
class State(TypedDict):
    task: str
    result: str
    judgment: str  # YES / NO
    iteration: int

# -----------------------
# TASK 1: Agent produces work
# -----------------------
def task_1_agent(state: State):
    prompt = f"""
You are performing a task.

Task: {state['task']}
Current result: {state['result']}

Improve or advance the result.
Return ONLY the updated result.
"""
    if state['iteration'] is None:
        iteration = 1
    else:
        iteration = state['iteration'] + 1

    print("calling LLM model...")
    response = llm.invoke(prompt)
    print(f"result: {response.content}")
    return {"result": response.content, "iteration": iteration}

# -----------------------
# TASK 2: Agent judges completion
# -----------------------
def task_2_judge_agent(state: State):
    prompt = f"""
You are a strict evaluator.

Task: {state['task']}
Result: {state['result']}

Is the task COMPLETE?

Think carefully, but respond ONLY with YES or NO.
"""

    response = llm.invoke(prompt)
    print(f"judgement: {response.content}")
    return {"judgment": response.content.strip().upper()}


# -----------------------
# ROUTING LOGIC
# -----------------------
def route_after_tool(state: State):
    last_output = state.get("judgment", "")

    if state['iteration'] >= MaxIter:
        print(f"Max iterations reached: {MaxIter}")
        return END

    if "YES" in last_output:
        return END
    return "task_1"

# -----------------------
# BUILD GRAPH
# -----------------------
builder = StateGraph(State)

builder.add_node("task_1", task_1_agent)
builder.add_node("task_2", task_2_judge_agent)

builder.set_entry_point("task_1")

# Flow:
builder.add_edge("task_1", "task_2")

builder.add_conditional_edges(
    "task_2",
    route_after_tool,
    {
        "task_1": "task_1",
        END: END
    }
)

graph = builder.compile()

# -----------------------
# RUN
# -----------------------
graph.invoke({
    "task": "Keep improving this explanation forever. Make it clearer each time. It's never good enough",
    "result": "",
    "judgment": "",
    "iteration": 0
})